# Neo4j 카드 샘플 점검
로컬 Neo4j에서 카드 샘플을 점검합니다. 기본 연결은 localhost 전용이며 원격에는 쓰지 않습니다. `WRITE=False`가 기본값입니다. 확인 후 출력 지우기(Clear All Outputs) 후 저장하세요. PDF 원문은 표시하지 마세요.

# 경로와 옵션
기본 경로에서는 `.env`를 읽지 않습니다. 원격 읽기는 `USE_ENV_REMOTE=True`로 명시적으로 선택해야 합니다.

In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd()
SAMPLE_FILES = [Path('/tmp/fp-sample-hana.cypher'), Path('/tmp/fp-sample-woori.cypher')]
LOCAL_URI = 'bolt://localhost:7687'
LOCAL_USERNAME = 'neo4j'
USE_ENV_REMOTE = False
ALLOW_REMOTE = False
WRITE = False
CONFIRM_DELETE = False
TOP_CARD_ID = None
DATABASE = 'neo4j'
node_labels = ['FpIssuer', 'FpCard', 'FpBenefit', 'FpCondition', 'FpLimit', 'FpTarget', 'FpEvidence']
static_check_ok = False
statements_by_file = {}
existing_labels = set()
existing_relationship_types = set()
existing_property_keys = set()
is_local = False
writes_allowed = False

# 연결
기본은 `bolt://localhost:7687` 로컬 연결입니다. 빈 비밀번호를 입력하면 인증 없이 연결을 시도합니다. 접속 값은 출력하지 않습니다.

In [ ]:
import getpass
import os
from urllib.parse import urlparse
from neo4j import GraphDatabase

driver = None
connected = False
remote_host = False
if USE_ENV_REMOTE:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
    _uri = os.environ.get('NEO4J_URI', '')
    _username = os.environ.get('NEO4J_USERNAME', '')
    _password = os.environ.get('NEO4J_PASSWORD', '')
    DATABASE = os.environ.get('NEO4J_DATABASE', 'neo4j')
else:
    _uri = LOCAL_URI
    _username = LOCAL_USERNAME
    _password = getpass.getpass('로컬 Neo4j 비밀번호 (인증 없음이면 빈 입력): ')

_host = (urlparse(_uri).hostname or '').lower()
is_local = _host in {'localhost', '127.0.0.1', '::1', 'host.docker.internal'}
remote_host = not is_local
writes_allowed = is_local
print('호스트 종류:', '로컬' if is_local else '원격')
if remote_host:
    print('원격 호스트이므로 쓰기·삭제는 차단됩니다.')
if remote_host and not ALLOW_REMOTE:
    print('원격 읽기도 차단됩니다: ALLOW_REMOTE=False')
elif not _uri:
    print('연결 실패')
else:
    try:
        _auth = (_username, _password) if _password else None
        if remote_host and not ALLOW_REMOTE:
            raise RuntimeError('원격 읽기가 비활성화됨')
        driver = GraphDatabase.driver(_uri, auth=_auth)
        driver.verify_connectivity()
        connected = True
        print('연결 성공')
    except Exception:
        print('연결 실패')

# 기존 라벨 및 팀원 그래프 점검
팀원 `dp_` 라벨은 경고만 표시합니다. 쓰기 대상은 정적 검사로 `Fp` 라벨과 `FP_` 관계에 한정합니다.

In [ ]:
import pandas as pd

def refresh_schema():
    global labels_df, relationship_types_df, property_keys_df
    global existing_labels, existing_relationship_types, existing_property_keys
    labels_df = pd.DataFrame(columns=['label'])
    relationship_types_df = pd.DataFrame(columns=['relationshipType'])
    property_keys_df = pd.DataFrame(columns=['propertyKey'])
    if connected:
        with driver.session(database=DATABASE) as session:
            labels_df = pd.DataFrame([record.data() for record in session.run('CALL db.labels()')], columns=['label'])
            relationship_types_df = pd.DataFrame([record.data() for record in session.run('CALL db.relationshipTypes()')], columns=['relationshipType'])
            property_keys_df = pd.DataFrame([record.data() for record in session.run('CALL db.propertyKeys()')], columns=['propertyKey'])
    existing_labels = set(labels_df['label'].dropna())
    existing_relationship_types = set(relationship_types_df['relationshipType'].dropna())
    existing_property_keys = set(property_keys_df['propertyKey'].dropna())

refresh_schema()
dp_labels = sorted(label for label in existing_labels if label.startswith('dp_'))
if dp_labels:
    print('팀원 그래프가 있는 DB 입니다. Fp 라벨만 쓰므로 계속 가능하지만 확인하세요')
print('Fp 라벨 존재:', any(label.startswith('Fp') for label in existing_labels))
display(labels_df)
display(relationship_types_df)


# 샘플 파일 정적 검사
생성기 관계와 대조하며 타입 없는 관계·비허용 라벨/관계·위험 명령을 차단합니다.

In [ ]:
import re

allowed_relationships = {
    'FP_OFFERS', 'FP_HAS_CONDITION', 'FP_HAS_LIMIT', 'FP_APPLIES_TO',
    'FP_EXCLUDES', 'FP_MUTUALLY_EXCLUSIVE', 'FP_HAS_EVIDENCE',
}

def inspect_cypher(cypher):
    cypher = re.sub(r'(?m)^\s*//.*$', '', cypher)
    forbidden = []
    for pattern, name in [
        (r'(?i)\bDELETE\b', 'DELETE'),
        (r'(?i)\bDROP\b', 'DROP'),
        (r'(?i)\bREMOVE\b', 'REMOVE'),
        (r'(?i)\bCALL\b', 'CALL'),
        (r'(?i)\bAPOC\b', 'apoc'),
        (r'(?i)\bLOAD\s+CSV\b', 'LOAD CSV'),
        (r'(?i)\b(?:OPTIONAL\s+)?MATCH\s*\(\s*(?:[A-Za-z_]\w*\s*)?(?:\{[^}]*\}\s*)?\)', '라벨 없는 MATCH'),
    ]:
        if re.search(pattern, cypher):
            forbidden.append(name)

    used_labels = set()
    for node_pattern in re.findall(r'\(([^()]*)\)', cypher):
        header = node_pattern.split('{', 1)[0]
        if ':' in header:
            used_labels.update(re.findall(r':\s*([A-Za-z_]\w*)', header))
    other_labels = sorted(used_labels - set(node_labels))
    if other_labels:
        forbidden.append('비허용 라벨: ' + ', '.join(other_labels))

    used_relationships = set()
    untyped = False
    relationship_patterns = re.finditer(
        r'(?:-\[([^\]]*)\](?:->|-)|<-\[([^\]]*)\]-)', cypher
    )
    for match in relationship_patterns:
        relationship_pattern = match.group(1) if match.group(1) is not None else match.group(2)
        header = relationship_pattern.split('{', 1)[0]
        match = re.search(r':\s*([A-Za-z_]\w*)', header)
        if match:
            used_relationships.add(match.group(1))
        else:
            untyped = True
    other_relationships = sorted(used_relationships - allowed_relationships)
    if other_relationships:
        forbidden.append('비허용 관계: ' + ', '.join(other_relationships))
    if untyped:
        forbidden.append('유형 없는 관계')
    return {'금지 패턴': forbidden, '라벨': used_labels, '관계': used_relationships}

statements_by_file = {}
checks = []
for path in SAMPLE_FILES:
    text = path.read_text(encoding='utf-8') if path.exists() else ''
    result = inspect_cypher(text)
    statements = []
    current = []
    for line in re.sub(r'(?m)^\s*//.*$', '', text).splitlines():
        current.append(line)
        if line.rstrip().endswith(';'):
            statements.append('\n'.join(current).strip())
            current = []
    if ''.join(current).strip():
        statements.append('\n'.join(current).strip())
    statements_by_file[path] = statements
    forbidden = list(result['금지 패턴'])
    if not path.exists():
        forbidden.append('파일 없음')
    if not statements:
        forbidden.append('문장 없음')
    checks.append({'파일': path.name, '문장 수': len(statements), '라벨': ', '.join(sorted(result['라벨'])),
                   '관계': ', '.join(sorted(result['관계'])), '금지 패턴': ', '.join(forbidden) or '없음'})
inspection_df = pd.DataFrame(checks, columns=['파일', '문장 수', '라벨', '관계', '금지 패턴'])
display(inspection_df)
static_check_ok = bool(checks) and all(row['금지 패턴'] == '없음' for row in checks)  # '없음' 문자열은 비어 있지 않아 any() 로 판정하면 항상 차단된다

# 샘플 적재
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if not WRITE:
    print('건너뜀')
elif not connected or not is_local or not writes_allowed or not static_check_ok:
    print('적재 차단: 로컬 연결과 정적 검사를 확인하세요')
else:
    import time
    started = time.perf_counter()
    executed = 0
    failures = 0
    with driver.session(database=DATABASE) as session:
        for path, statements in statements_by_file.items():
            for statement in statements:
                try:
                    session.run(statement).consume()
                    executed += 1
                except Exception:
                    failures += 1
                    print('문장 실행 실패:', path.name)
                    break
            if failures:
                break
    refresh_schema()
    print(f'실행 문장 수: {executed}, 실패 문장 수: {failures}, 소요 시간: {time.perf_counter()-started:.2f}초')


# Fp 그래프 확인 질의
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if connected:
    with driver.session(database=DATABASE) as session:
        def query_df(query, **params):
            return pd.DataFrame([record.data() for record in session.run(query, **params)])

        def count_nodes(where=''):
            counts = {}
            for label in node_labels:
                if label in existing_labels:
                    row = query_df(
                        f"MATCH (n:{label}){where} RETURN count(n) AS count"
                    ).iloc[0]
                    counts[label] = row['count']
                else:
                    counts[label] = 0
            return pd.DataFrame([{'label': label, 'count': counts[label]} for label in node_labels])

        node_counts = count_nodes()
        rel_counts = query_df(
            "MATCH ()-[r]->() WHERE type(r) STARTS WITH 'FP_' "
            "RETURN type(r) AS relationship, count(r) AS count ORDER BY relationship"
        )
        cards_benefits = query_df(
            "MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) "
            "RETURN c.name AS card, count(DISTINCT b) AS benefits ORDER BY benefits DESC"
        ) if {'FpCard', 'FpBenefit'} <= existing_labels and 'FP_OFFERS' in existing_relationship_types and 'name' in existing_property_keys else pd.DataFrame(columns=['card', 'benefits'])
        duplicate_benefits = query_df(
            "MATCH (b:FpBenefit) WITH b.name AS name, collect(b.id) AS ids, count(*) AS copies "
            "WHERE copies > 1 RETURN name, ids, copies ORDER BY copies DESC LIMIT 20"
        ) if 'FpBenefit' in existing_labels and {'id', 'name'} <= existing_property_keys else pd.DataFrame(columns=['name', 'ids', 'copies'])
        multi_card_benefits = query_df(
            "MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) "
            "WITH b, count(DISTINCT c) AS cards WHERE cards > 1 "
            "RETURN b.id AS id, b.name AS name, cards ORDER BY cards DESC LIMIT 10"
        ) if {'FpCard', 'FpBenefit'} <= existing_labels and 'FP_OFFERS' in existing_relationship_types and {'id', 'name'} <= existing_property_keys else pd.DataFrame(columns=['id', 'name', 'cards'])
        evidence_status = query_df(
            "MATCH (e:FpEvidence) RETURN e.quote_status AS quote_status, count(e) AS count"
        ) if 'FpEvidence' in existing_labels and 'quote_status' in existing_property_keys else pd.DataFrame(columns=['quote_status', 'count'])
        missing_names = (
            count_nodes(" WHERE n.name IS NULL OR n.name = ''")
            if 'name' in existing_property_keys else node_counts.copy()
        )

    for title, frame in [
        ('Fp 라벨별 노드 수', node_counts), ('FP_ 관계별 수', rel_counts),
        ('카드별 혜택 수', cards_benefits), ('동일 이름 혜택 상위 20', duplicate_benefits),
        ('여러 카드 연결 혜택 상위 10', multi_card_benefits),
        ('근거 quote_status별 수', evidence_status), ('name 누락 노드 수', missing_names),
    ]:
        print(title)
        display(frame)


# 최대 2단계 그래프 그림
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import font_manager

required_labels = {'FpCard', 'FpBenefit'}
if connected and required_labels <= existing_labels and 'FP_OFFERS' in existing_relationship_types and 'id' in existing_property_keys:
    with driver.session(database=DATABASE) as session:
        card = session.run(
            "MATCH (c:FpCard)-[:FP_OFFERS]->(b:FpBenefit) "
            "RETURN c.id AS id, count(DISTINCT b) AS n ORDER BY n DESC, id LIMIT 1"
        ).single() if TOP_CARD_ID is None else {'id': TOP_CARD_ID}
        graph_rows = list(session.run(
            "MATCH (c:FpCard {id:$id})-[r1:FP_OFFERS]-(b:FpBenefit) "
            "OPTIONAL MATCH (b)-[r2]-(d) "
            "WHERE type(r2) IN $detail_types "
            "AND any(label IN labels(d) WHERE label IN $detail_labels) "
            "RETURN c,b,d,r1,r2 LIMIT 80",
            id=card['id'],
            detail_types=['FP_HAS_CONDITION', 'FP_HAS_LIMIT', 'FP_APPLIES_TO', 'FP_EXCLUDES'],
            detail_labels=['FpCondition', 'FpLimit', 'FpTarget'],
        )) if card else []

    graph = nx.Graph()
    for row in graph_rows:
        for node in (row['c'], row['b'], row['d']):
            if node is not None:
                label = next(label for label in node.labels if label in node_labels)
                graph.add_node(node['id'], label=label, name=node.get('name') or node['id'])
        for relationship in (row['r1'], row['r2']):
            if relationship is not None:
                graph.add_edge(
                    relationship.start_node['id'], relationship.end_node['id'],
                    label=relationship.type,
                )

    candidates = ['Noto Sans CJK KR', 'NanumGothic', 'Malgun Gothic']
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font = next((name for name in candidates if name in installed), None)
    if font:
        plt.rcParams['font.family'] = font

    colors = {
        'FpCard': '#4C78A8', 'FpBenefit': '#F58518', 'FpCondition': '#54A24B',
        'FpLimit': '#E45756', 'FpTarget': '#B279A2', 'FpIssuer': '#72B7B2',
        'FpEvidence': '#FF9DA6',
    }
    pos = nx.spring_layout(graph, seed=7)
    for label, color in colors.items():
        nodes = [node for node, data in graph.nodes(data=True) if data['label'] == label]
        if nodes:
            nx.draw_networkx_nodes(graph, pos, nodelist=nodes, node_color=color, label=label, node_size=900)
    labels = {
        node: data['name'][:12] if font else f"{data['label'][-4:]}:{node[-6:]}"
        for node, data in graph.nodes(data=True)
    }
    nx.draw_networkx_edges(graph, pos, alpha=.45)
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)
    plt.legend()
    plt.axis('off')
    plt.show()
elif connected:
    print('그래프 그림 건너뜀: FpCard/FpBenefit/FP_OFFERS가 DB에 없습니다.')


# Fp 데이터 정리
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if remote_host or not is_local:
    print('삭제 셀 차단: 원격 호스트에서는 삭제 셀을 실행하지 않습니다')
elif not CONFIRM_DELETE:
    if connected:
        with driver.session(database=DATABASE) as session:
            result = session.run(
                "MATCH (n) "
                "WHERE size(labels(n)) = 1 AND labels(n)[0] IN $labels "
                "RETURN count(n) AS count",
                labels=node_labels,
            ).single()
        print('삭제 대상 Fp 노드 수:', result['count'])
    else:
        print('미리보기에는 연결이 필요합니다')
elif not connected or not is_local or not writes_allowed:
    print('삭제 차단: 로컬 연결만 허용됩니다')
else:
    with driver.session(database=DATABASE) as session:
        result = session.run(
            "MATCH (n) "
            "WHERE size(labels(n)) = 1 AND labels(n)[0] IN $labels "
            "DETACH DELETE n",
            labels=node_labels,
        )
        result.consume()
    refresh_schema()
    print('Fp 노드 정리 완료')


# 연결 닫기
이 셀은 설명된 단계만 수행합니다. 출력에는 접속 비밀이나 PDF 원문을 표시하지 않습니다.

In [ ]:
if driver is not None:
    driver.close()
    print('연결 닫음')
else:
    print('열린 연결 없음')
